# Airport Runway FOD Detection — YOLOv8n Baseline Training

This Colab notebook trains the reportable original-image baseline using the frozen leakage-resistant split. It reproduces the local conversion and training configuration; it does not create a new random split.

Before running, select **Runtime → Change runtime type → T4 GPU** (or another available GPU). Upload `FODPascalVOCFormat-V.2.1.zip` to Google Drive.

In [ ]:
# Confirm that Colab assigned a CUDA GPU.
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable a GPU from Runtime > Change runtime type before continuing.'
print('CUDA device:', torch.cuda.get_device_name(0))

## 1. Mount Google Drive and configure paths

Change `DRIVE_DATASET_ZIP` if the uploaded ZIP is stored in another Drive folder. Training outputs will be copied to `DRIVE_RESULTS_DIR`.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

DRIVE_DATASET_ZIP = Path('/content/drive/MyDrive/Machine Vision/FODPascalVOCFormat-V.2.1.zip')
DRIVE_RESULTS_DIR = Path('/content/drive/MyDrive/Machine Vision/FOD Baseline Results')
REPOSITORY_DIR = Path('/content/airport-runway-fod-detection')
assert DRIVE_DATASET_ZIP.exists(), f'Dataset ZIP not found: {DRIVE_DATASET_ZIP}'
print('Dataset ZIP:', DRIVE_DATASET_ZIP)
print('Results destination:', DRIVE_RESULTS_DIR)

## 2. Obtain the version-controlled project

The exact `baseline-model-training` branch is used so the frozen split, converter and training settings match the audited project.

In [ ]:
import shutil
import subprocess

if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
subprocess.run([
    'git', 'clone', '--branch', 'baseline-model-training', '--single-branch',
    'https://github.com/neelia26/airport-runway-fod-detection.git',
    str(REPOSITORY_DIR),
], check=True)
%cd /content/airport-runway-fod-detection
!git log -1 --oneline

## 3. Install the training environment

In [ ]:
%pip install -q -e '.[training]'
!python scripts/check_training_environment.py

## 4. Extract the untouched Pascal VOC source dataset

Extraction occurs in Colab's temporary storage. The ZIP in Drive is read but not modified.

In [ ]:
import zipfile

dataset_parent = REPOSITORY_DIR / 'FOD-A Dataset'
dataset_parent.mkdir(parents=True, exist_ok=True)
expected_voc_root = dataset_parent / 'FODPascalVOCFormat-V.2.1' / 'VOC2007'
if not expected_voc_root.exists():
    with zipfile.ZipFile(DRIVE_DATASET_ZIP) as archive:
        archive.extractall(dataset_parent)
assert expected_voc_root.exists(), f'Unexpected ZIP structure: {expected_voc_root} not found'
print('Pascal VOC dataset:', expected_voc_root)

## 5. Reproduce the verified single-class YOLO conversion

This uses the committed sequence-aware train, validation and test manifests. All 31 source categories are mapped to class `0: FOD`.

In [ ]:
!MPLBACKEND=Agg python scripts/convert_voc_to_yolo.py
!python scripts/verify_yolo_conversion.py

## 6. Optional pipeline smoke test

This one-epoch subset run is only a technical check. Its metrics must not be reported as research results.

In [ ]:
!python scripts/train_baseline.py --smoke-test --device cuda

## 7. Train the reportable baseline

The command below trains YOLOv8n for at most 30 epochs with early stopping. Do not change the split, image size, batch size or seed if this run will be compared with later preprocessing variants.

In [ ]:
!python scripts/train_baseline.py --device cuda

## 8. Preserve the complete run in Google Drive

Colab storage is temporary. This cell copies checkpoints, metrics and figures to Drive after training.

In [ ]:
from datetime import datetime

source_run = REPOSITORY_DIR / 'runs' / 'baseline' / 'yolov8n_original'
assert source_run.exists(), f'Completed run not found: {source_run}'
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
destination = DRIVE_RESULTS_DIR / f'yolov8n_original_{timestamp}'
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(source_run, destination)
print('Saved complete baseline run to:', destination)

## Next step

Do not interpret the final metrics in isolation. The saved `best.pt` checkpoint must next be evaluated on the frozen test partition by object size, lighting and weather. Gaussian and median variants should only be trained after the original-image baseline has been verified.